In [ ]:
import rasterio
import numpy as np
import glob
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# =========================
# 1. 文件路径
# =========================
all_files = [str(p) for p in config.RAW_NO_CA]

print("GRD数量:", len(all_files))

for f in all_files:
    print(f)

# =========================
# 2. reference grid
# =========================
ref_path = all_files[0]

with rasterio.open(ref_path) as ref:
    ref_shape = (ref.height, ref.width)
    profile = ref.profile.copy()

    h = ref.height
    w = ref.width

print("参考栅格尺寸:", h, w)

# =========================
# 3. 栅格对齐函数
# =========================
def align_no_crs(f):

    with rasterio.open(f) as src:

        data = src.read(1).astype(np.float32)

        out = np.full(ref_shape, np.nan, dtype=np.float32)

        hh = min(h, data.shape[0])
        ww = min(w, data.shape[1])

        out[:hh, :ww] = data[:hh, :ww]

    return out

# =========================
# 4. 构建特征矩阵 X
# =========================
features = []

for f in all_files:

    arr = align_no_crs(f)

    features.append(arr.flatten())

X = np.column_stack(features)

print("X shape:", X.shape)

# =========================
# 5. 处理Surfer NoData
# =========================
X[np.abs(X) > 1e30] = np.nan

# =========================
# 6. 读取矿点CSV
# =========================
mine_df = pd.read_csv(
    config.SAMPLE_PUL,
    header=None,
    names=["x", "y", "label"]
)

labels = np.zeros(X.shape[0], dtype=int)

with rasterio.open(ref_path) as src:

    width = src.width
    height = src.height

    hit = 0

    for _, row in mine_df.iterrows():

        x = float(row["x"])
        y = float(row["y"])

        try:

            r, c = src.index(x, y)

            if 0 <= r < height and 0 <= c < width:

                idx = r * width + c

                labels[idx] = 1

                hit += 1

            else:

                print("越界:", x, y)

        except Exception as e:

            print("失败:", x, y, e)

print("成功写入矿点数:", hit)
print("labels sum:", labels.sum())

# =========================
# 7. 去除NaN像元
# =========================
mask = ~np.isnan(X).any(axis=1)

X_clean = X[mask]
labels_clean = labels[mask]

P_index = np.where(labels_clean == 1)[0]
U_index = np.where(labels_clean == 0)[0]

print("P样本:", len(P_index))
print("U样本:", len(U_index))

# =========================
# 8. PU Bagging RF
# =========================
T = 100

nP = len(P_index)

pred_matrix = np.zeros((X_clean.shape[0], T))

for t in range(T):

    sizeU = min(nP, len(U_index))

    sampled_U = np.random.choice(
        U_index,
        size=sizeU,
        replace=False
    )

    train_idx = np.concatenate([
        P_index,
        sampled_U
    ])

    X_train = X_clean[train_idx]

    y_train = np.concatenate([
        np.ones(len(P_index)),
        np.zeros(len(sampled_U))
    ])

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=t,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    pred_matrix[:, t] = rf.predict_proba(X_clean)[:, 1]

print("PU训练完成")

# =========================
# 9. 概率与不确定性
# =========================
pu_prob = pred_matrix.mean(axis=1)

pu_std = pred_matrix.std(axis=1)

print("概率范围:")
print(
    np.nanmin(pu_prob),
    np.nanmax(pu_prob)
)

print("不确定性范围:")
print(
    np.nanmin(pu_std),
    np.nanmax(pu_std)
)

# =========================
# 10. 回填栅格
# =========================
full_prob = np.full(X.shape[0], np.nan)

full_std = np.full(X.shape[0], np.nan)

full_prob[mask] = pu_prob

full_std[mask] = pu_std

pu_map = full_prob.reshape(h, w)

std_map = full_std.reshape(h, w)

# =========================
# 11. 填补空洞
# =========================
valid_mask = ~np.isnan(pu_map)

if np.any(~valid_mask):

    pu_map[~valid_mask] = np.interp(
        np.flatnonzero(~valid_mask),
        np.flatnonzero(valid_mask),
        pu_map[valid_mask]
    )

valid_mask_std = ~np.isnan(std_map)

if np.any(~valid_mask_std):

    std_map[~valid_mask_std] = np.interp(
        np.flatnonzero(~valid_mask_std),
        np.flatnonzero(valid_mask_std),
        std_map[valid_mask_std]
    )

# =========================
# 12. 输出GeoTIFF
# =========================
profile.update(
    dtype="float32",
    count=1,
    nodata=-9999,
    compress="lzw"
)

out_dir = config.ensure_dir(config.OUTPUT / "pu_rf")
out_prob = str(out_dir / "PU_RF_result.tif")
out_std = str(out_dir / "PU_RF_uncertainty.tif")

with rasterio.open(
    out_prob,
    "w",
    **profile
) as dst:

    tmp = pu_map.copy()

    tmp[np.isnan(tmp)] = -9999

    dst.write(
        tmp.astype(np.float32),
        1
    )

with rasterio.open(
    out_std,
    "w",
    **profile
) as dst:

    tmp = std_map.copy()

    tmp[np.isnan(tmp)] = -9999

    dst.write(
        tmp.astype(np.float32),
        1
    )

print("================================")
print("PU-RF完成")
print("概率图：", out_prob)
print("不确定性图：", out_std)
print("================================")

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

X_shap = X_clean.copy()
y_shap = labels_clean.copy()

print("X shape:", X_shap.shape)
print("y shape:", y_shap.shape)

feature_names = [
    "Pb", "Zn", "Cu", "Ag",
    "Au", "As", "Hg", "Sb",
    "Ip-M", "Ip-ρ", "Magnetic"
]

rf_shap = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_shap.fit(X_shap, y_shap)

print("SHAP RF训练完成")

explainer = shap.TreeExplainer(rf_shap)

shap_values = explainer(X_shap)

print("SHAP计算完成")

#处理维度
sv = shap_values.values

print("SHAP原始维度:", sv.shape)

# 二分类情况处理
if len(sv.shape) == 3:
    sv = sv[:, :, 1]   # 取“成矿类”

print("修正后SHAP维度:", sv.shape)

#全局重要性图
shap.summary_plot(
    sv,
    X_shap
)

#变量重要性（条形图）
shap.summary_plot(
    sv,
    X_shap,
    plot_type="bar"
)

#单变量影响
pb_index = feature_names.index("Pb")

shap.dependence_plot(
    pb_index,
    sv,
    X_shap
)


In [ ]:
import pandas as pd
import numpy as np
import os

#确保SHAP矩阵
sv = shap_values.values

# 二分类处理
if len(sv.shape) == 3:
    sv = sv[:, :, 1]

print("SHAP shape:", sv.shape)

#导出总SHAP表
df_shap = pd.DataFrame(
    sv,
    columns=feature_names
)

df_shap.to_csv(
    str(config.OUTPUT / "pu_rf_shap" / "SHAP_all_variables.csv"),
    index=False
)

print("已保存：SHAP_all_variables.csv")

#逐变量导出（单个文件🔥）
out_dir = str(config.OUTPUT / "pu_rf_shap" / "SHAP_single_vars")
os.makedirs(out_dir, exist_ok=True)

for i, name in enumerate(feature_names):

    df = pd.DataFrame({
        name: sv[:, i]
    })

    df.to_csv(
        os.path.join(out_dir, f"{name}_SHAP.csv"),
        index=False
    )

print("单变量SHAP已全部导出")

#导出“空间版本”（可用于Surfer/GMT）

full_shap = np.full((X.shape[0], sv.shape[1]), np.nan)

for i in range(len(feature_names)):
    full_shap[mask, i] = sv[:, i]


#转CSV（带像元索引）
df_grid = pd.DataFrame(full_shap, columns=feature_names)

df_grid.to_csv(
    str(config.OUTPUT / "pu_rf_shap" / "SHAP_grid_all.csv"),
    index=False
)

print("空间SHAP已导出")